### Output Guardrails

In [1]:
from agents import Agent, Runner, GuardrailFunctionOutput, OutputGuardrail, OutputGuardrailTripwireTriggered
from pydantic import BaseModel, Field

##### Basic Output Guardrail

In [2]:
class MathOutput(BaseModel):
    is_math: bool = Field(description="True if the input is a math-related question, False otherwise")
    reasoning: str = Field(description="Short justification of why it is or isn’t math-related")

guardrail_agent = Agent(
    name="Guardrail check",
    instructions="Check if the input includes any math",
    output_type=MathOutput,
    model="gpt-4.1-nano"
)

async def math_guardrail(ctx, agent, output):
    result = await Runner.run(guardrail_agent, output, context=ctx.context)

    return GuardrailFunctionOutput(
        output_info=result.final_output.reasoning,
        tripwire_triggered=result.final_output.is_math,
    )

In [3]:
agent = Agent(
    name="Agent",
    instructions="You are a helpful agent",
    model="gpt-4.1-nano",
    output_guardrails=[
        OutputGuardrail(guardrail_function=math_guardrail),
    ]
)

async def run_agent(prompt):
    try:
        result = await Runner.run(agent, prompt)
        print(result.final_output)

    except OutputGuardrailTripwireTriggered as e:
         print("OutputGuardrailTripwireTriggered exception caught: ", e.guardrail_result.output.output_info)

In [6]:
await run_agent("Quanto fa 2x2?")

OutputGuardrailTripwireTriggered exception caught:  The input '2 x 2 fa 4' involves multiplication, which is a mathematical operation, indicating that it is math-related.


##### Output Guardrail con Handoff Diretto

In [12]:
math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
    output_guardrails=[
        OutputGuardrail(guardrail_function=math_guardrail)
    ],
    model="gpt-4.1-nano"
)


starting_agent = Agent(
    name="Triage Agent",
    instructions="Handoff to the proper agent to answer the user question",
    handoffs=[math_tutor_agent],
    model="gpt-4.1-nano",
    output_guardrails=[
        OutputGuardrail(guardrail_function=math_guardrail)
    ]
)

async def run_agent(prompt):
    try:
        result = await Runner.run(starting_agent, prompt)
        print(result.final_output)

    except OutputGuardrailTripwireTriggered as e:
         print("OutputGuardrailTripwireTriggered exception caught: ", e.guardrail_result.output.output_info)

In [13]:
await run_agent("Quanto fa 2x2?")

OutputGuardrailTripwireTriggered exception caught:  The input discusses multiplication and addition, which are mathematical concepts and operations.


##### Output Guardrail con Handoff Agents as Tools

In [16]:
math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
    model="gpt-4.1-nano"
)

math_tool = math_tutor_agent.as_tool(
    tool_name="ask_mathematical_agent",
    tool_description="Return an answer to any mathematical questions"
)

In [17]:
agent_with_tool = Agent(
    name="Test Output Guardrail with Agents as Tools",
    instructions="You are a useful assistant. You can use the available tools to manage user questions",
    tools=[math_tool],
    model="gpt-4.1-nano",
    output_guardrails=[
        OutputGuardrail(guardrail_function=math_guardrail)
    ]
)

async def run_agent(prompt):
    try:
        result = await Runner.run(agent_with_tool, prompt)
        print(result.final_output)

    except OutputGuardrailTripwireTriggered as e:
         print("OutputGuardrailTripwireTriggered exception caught: ", e.guardrail_result.output.output_info)

In [18]:
await run_agent("Quanto fa 2x2?")

OutputGuardrailTripwireTriggered exception caught:  The input includes a mathematical expression '2 x 2' and its result, which makes it math-related.
